# Credit Risk Model: PD, LGD, EAD, and Expected Loss

Credit Risk Model: PD, LGD, EAD, and Expected Loss (SIMPLE VERSION)
--------------------------------------------------------------------
Dataset: German Credit Data (Statlog, UCI / Kaggle "german-credit-data-with-risk")
Download from Kaggle (search "German Credit Data with Risk", uploader kabure)
and save it as "german_credit_data.csv" in this same folder.

This version avoids advanced sklearn machinery (no Pipeline, no
ColumnTransformer). It uses only:
  - pandas for data handling
  - pd.get_dummies() to turn text categories into numbers (0/1 columns)
  - one LogisticRegression model
  - matplotlib for two simple charts

Read every block below - each one does ONE thing. If a block confuses you,
that's exactly what to ask about before you say you "built" this project.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve


## STEP 1: Load the data


In [2]:
df = pd.read_csv("german_credit_data.csv")
print(df.columns.tolist())
df.head()

[',Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk']


,",Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk"
0,"0,67,male,2,own,NA,little,1169,6,radio/TV,good"
1,"1,22,female,2,own,little,moderate,5951,48,radi..."
2,"2,49,male,1,own,little,NA,2096,12,education,good"
3,"3,45,male,2,free,little,little,7882,42,furnitu..."
4,"4,53,male,2,free,little,little,4870,24,car,bad"


In [3]:
from io import StringIO

# Your file has each row wrapped in an extra pair of quotes, plus a
# leading index column with no name - this cleans both issues up.
with open("german_credit_data.csv", encoding="utf-8-sig") as f:
    lines = [line.strip().strip('"') for line in f]
csv_text = "\n".join(lines)

df = pd.read_csv(StringIO(csv_text), index_col=0)

# The "Risk" column says 'good' or 'bad'. We turn that into a number:
# 1 = this person defaulted ("bad"), 0 = they did not ("good").
df["default"] = (df["Risk"].str.lower() == "bad").astype(int)

print("Rows, columns:", df.shape)
print(df.head())
print("\nShare of loans that defaulted:", round(df["default"].mean(), 3))

Rows, columns: (1000, 11)
   Age     Sex  Job Housing Saving accounts Checking account  Credit amount  \
0   67    male    2     own             NaN           little           1169   
1   22  female    2     own          little         moderate           5951   
2   49    male    1     own          little              NaN           2096   
3   45    male    2    free          little           little           7882   
4   53    male    2    free          little           little           4870   

   Duration              Purpose  Risk  default  
0         6             radio/TV  good        0  
1        48             radio/TV   bad        1  
2        12            education  good        0  
3        42  furniture/equipment  good        0  
4        24                  car   bad        1  

Share of loans that defaulted: 0.3


## STEP 2: Look at where risk concentrates (simple groupby, no modeling yet)


In [4]:
print("\nDefault rate by loan purpose:")
print(df.groupby("Purpose")["default"].mean().round(3).sort_values(ascending=False))

print("\nDefault rate by housing situation:")
print(df.groupby("Housing")["default"].mean().round(3).sort_values(ascending=False))



Default rate by loan purpose:
Purpose
vacation/others        0.417
education              0.390
repairs                0.364
business               0.351
domestic appliances    0.333
furniture/equipment    0.320
car                    0.315
radio/TV               0.221
Name: default, dtype: float64

Default rate by housing situation:
Housing
free    0.407
rent    0.391
own     0.261
Name: default, dtype: float64


## STEP 3: Prepare features


In [5]:
# Pick a handful of columns to predict default with.
# Numeric columns can go into the model as-is.
numeric_cols = ["Age", "Job", "Credit amount", "Duration"]

# Text columns (like "Sex", "Housing") can't go into a model directly.
# pd.get_dummies() turns each category into its own 0/1 column.
# Example: "Housing" with values own/rent/free becomes 3 columns:
#   Housing_own, Housing_rent, Housing_free (each 0 or 1)
categorical_cols = ["Sex", "Housing", "Saving accounts", "Checking account", "Purpose"]
dummies = pd.get_dummies(df[categorical_cols], drop_first=True)

# Numeric columns like "Credit amount" (hundreds to thousands) and "Age"
# (tens) are on very different scales, which makes the model slower to
# train and can throw a harmless "failed to converge" warning. The simple
# fix: rescale each numeric column so it has mean 0 and spread (std) 1.
# This is just: (value - column_average) / column_spread
numeric_scaled = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std()

# Combine numeric + dummy columns into one feature table
X = pd.concat([numeric_scaled, dummies], axis=1)
y = df["default"]

print("\nNumber of features after get_dummies:", X.shape[1])



Number of features after get_dummies: 19


## STEP 4: Split into training data and test data


In [6]:
# We train the model on 75% of loans, and check how well it predicts
# the other 25% it has NEVER seen. This is how you tell if a model
# actually learned something, versus just memorizing the data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


## STEP 5: Train a Logistic Regression model


In [7]:
# Logistic Regression is the standard first model for PD (probability of
# default) because banks/regulators need models that are explainable -
# you can look at its coefficients and say "higher credit amount = higher
# risk" in plain language, unlike a black-box model.
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

# predict_proba gives a probability between 0 and 1 for each loan.
# [:, 1] takes the probability of class "1" (default).
pd_pred = model.predict_proba(X_test)[:, 1]


## STEP 6: Check how good the model is


In [8]:
# AUC (Area Under the ROC Curve) measures how well the model ranks
# risky vs. safe loans. 0.5 = no better than a coin flip. 1.0 = perfect.
# Real credit scorecards typically land around 0.65-0.80.
auc = roc_auc_score(y_test, pd_pred)
print(f"\nModel AUC: {auc:.3f}")

# Plot the ROC curve (a standard chart for showing this)
fpr, tpr, _ = roc_curve(y_test, pd_pred)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC={auc:.2f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - PD Model")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=150)
plt.close()



Model AUC: 0.670


## STEP 7: Turn PD into Expected Loss


In [9]:
# Expected Loss = PD x LGD x EAD
#   PD  = probability of default (from our model, above)
#   LGD = Loss Given Default - if they default, what % of the loan is lost?
#         This dataset has no collateral/recovery info, so we use a
#         standard assumption: 45%, which is Basel's Foundation-IRB
#         supervisory value for unsecured retail-type exposures.
#   EAD = Exposure at Default - how much money is at risk. We simplify
#         this to the full credit amount (no amortization modeled).
LGD = 0.45

results = X_test.copy()
results["PD"] = pd_pred
results["EAD"] = df.loc[X_test.index, "Credit amount"]
results["LGD"] = LGD
results["Expected_Loss"] = results["PD"] * results["LGD"] * results["EAD"]

total_exposure = results["EAD"].sum()
total_EL = results["Expected_Loss"].sum()

print(f"\nTotal exposure (test set):  {total_exposure:,.0f}")
print(f"Total expected loss:        {total_EL:,.0f}")
print(f"Expected loss / exposure:   {100 * total_EL / total_exposure:.2f}%")



Total exposure (test set):  861,392
Total expected loss:        204,984
Expected loss / exposure:   23.80%


## STEP 8: Show where the risk is concentrated


In [10]:
# Split loans into 5 risk groups (lowest PD to highest PD) and see
# how much of the total expected loss comes from each group.
results["risk_group"] = pd.qcut(results["PD"], 5, labels=[
    "1 - Lowest risk", "2", "3", "4", "5 - Highest risk"
])
summary = results.groupby("risk_group", observed=True).agg(
    avg_PD=("PD", "mean"),
    n_loans=("PD", "size"),
    total_EL=("Expected_Loss", "sum"),
).round(3)
print("\nExpected loss by risk group:")
print(summary)

plt.figure(figsize=(6, 4))
summary["total_EL"].plot(kind="bar", color="steelblue")
plt.title("Expected Loss by Risk Group")
plt.ylabel("Expected Loss")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("el_by_risk_group.png", dpi=150)
plt.close()

print("\nDone. Saved: roc_curve.png, el_by_risk_group.png")



Expected loss by risk group:
                  avg_PD  n_loans   total_EL
risk_group                                  
1 - Lowest risk    0.268       50  12820.344
2                  0.376       50  22815.847
3                  0.450       50  35905.683
4                  0.559       50  44267.437
5 - Highest risk   0.729       50  89174.232

Done. Saved: roc_curve.png, el_by_risk_group.png
